In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import anndata as ad
import numpy as np
import pandas as pd

from rich import print
import os

import scvi

import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import colorcet as cc
import sys

# from scib_metrics.benchmark import Benchmarker, BatchCorrection, BioConservation
from sklearn.cluster import KMeans

In [4]:
# Add to the system path
sys.path.append(os.path.abspath("../breast_original"))

from params import setup  # Now this should work!

from params import setup, niche_setup

In [5]:
sc.set_figure_params(dpi=150, fontsize=12, format="svg")
figure_dir = setup.FIGURES_FOLDER

plots = True

In [6]:
scvi.settings.seed = 34

Seed set to 34


In [7]:
data_dir, data_file = setup.DATA_FOLDER, setup.DATA_FILE

data_file_name = os.path.splitext(data_file)[0]
print(data_file_name)

path_to_save = "/home/nathanl/scviva_paper/breast/checkpoints/xenium_breast_cancer_S1_R1_2"

xenium_breast_cancer_S1_R1_2

In [ ]:
# Endothelial reference for Table 1: the Wu et al. 2021 atlas restricted to genes expressed in >=1% of
# endothelial cells, with the endothelial-vs-rest t-test. Built once, then read from ATLAS_DE_PATH.
ATLAS_PATH = "/home/nathanl/Data/breast_sc_atlas_wu2021.h5ad"
ATLAS_DE_PATH = "/home/nathanl/Data/breast_sc_atlas_wu2021_DE.h5ad"
if not os.path.exists(ATLAS_DE_PATH):
    _ref = ad.read_h5ad(ATLAS_PATH)
    _endo = _ref[_ref.obs["cell_type"] == "endothelial cell"].copy()
    sc.pp.filter_genes(_endo, min_cells=max(1, int(0.01 * _endo.n_obs)))
    _ref = _ref[:, _endo.var_names].copy()
    sc.tl.rank_genes_groups(
        _ref,
        groupby="cell_type",
        groups=["endothelial cell"],
        method="t-test",
        key_added="rank_genes_groups_ttest_cell_type",
        use_raw=False,
        pts=True,
        n_genes=None,
    )
    _ref.write_h5ad(ATLAS_DE_PATH)
    del _ref, _endo

In [8]:
# adata_ref = ad.read_h5ad("../breast_cancer/breast_sc_atlas.h5ad")
adata_ref = ad.read_h5ad(ATLAS_DE_PATH)
print(adata_ref)

AnnData object with n_obs × n_vars = 100064 × 12716
    obs: 'donor_id', 'percent_mito', 'nCount_RNA', 'nFeature_RNA', 'celltype_major', 'celltype_minor', 
'celltype_subset', 'subtype', 'gene_module', 'calls', 'normal_cell_call', 'CNA_value', 'batch_run', 'multiplexed', 
'cryo_state', 'development_stage_ontology_term_id', 'cancer_type', 'ER', 'PR', 'HER2_IHC', 'HER2_ISH', 
'HER2_ISH_ratio', 'Ki67', 'subtype_by_IHC', 'treatment_status', 'treatment_details', 'assay_ontology_term_id', 
'organism_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'sex_ontology_term_id', 
'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'disease_ontology_term_id', 'grade', 
'cell_type_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 
'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 
'feature_type'
    uns: 'citation', 'rank_genes_groups_ttest_cell_type', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_Three-D', 'X_umap'

In [68]:
# # Select only endothelial cells
# adata_sub = adata_ref[adata_ref.obs["cell_type"] == "endothelial cell"]

# # Compute min_cells as 1% of endothelial cells (rounded up)
# min_cells = max(1, int(0.01 * adata_sub.n_obs))

# # Apply gene filtering on the subset
# sc.pp.filter_genes(adata_sub, min_cells=min_cells)

# # Get the list of genes that passed the filtering
# filtered_genes = adata_sub.var_names

# # Filter the original dataset to retain only those genes
# adata_ref = adata_ref[:, filtered_genes].copy()

# adata_ref

In [69]:
adata_ref.X[0].toarray().max()

5.457769

In [70]:
adata_ref.obs["cancer_type"].value_counts()

cancer_type
IDC    80121
ILC    12570
MBC     7373
Name: count, dtype: int64

In [71]:
adata_ref.obs["cell_type"].value_counts()

cell_type
CD4-positive, alpha-beta T cell              19231
luminal epithelial cell of mammary gland     14367
mammary gland epithelial cell                13379
CD8-positive, alpha-beta T cell              11487
endothelial cell                              7402
fibroblast of breast                          6573
macrophage                                    5929
pericyte                                      5423
plasmablast                                   3524
memory B cell                                 2581
monocyte                                      2328
natural killer cell                           1846
T cell                                        1528
myeloid cell                                  1418
mature NK T cell                              1122
basal-myoepithelial cell of mammary gland     1098
naive B cell                                   625
endothelial cell of lymphatic vessel           203
Name: count, dtype: int64

In [72]:
adata_ref.var

,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000238009,False,ENSG00000238009.6,NCBITaxon:9606,gene,629,lncRNA
ENSG00000279928,False,DDX11L17,NCBITaxon:9606,gene,570,unprocessed_pseudogene
ENSG00000279457,False,WASH9P,NCBITaxon:9606,gene,1397,unprocessed_pseudogene
ENSG00000228463,False,ENSG00000228463.10,NCBITaxon:9606,gene,1902,transcribed_processed_pseudogene
ENSG00000237094,False,ENSG00000237094.13,NCBITaxon:9606,gene,183,transcribed_unprocessed_pseudogene
...,...,...,...,...,...,...
ENSG00000256286,False,ENSG00000256286.2,NCBITaxon:9606,gene,573,lncRNA
ENSG00000260268,False,LINC00919,NCBITaxon:9606,gene,1547,lncRNA
ENSG00000267047,False,ENSG00000267047.1,NCBITaxon:9606,gene,498,lncRNA
ENSG00000239886,False,KRTAP9-2,NCBITaxon:9606,gene,1005,protein_coding


In [73]:
# sc.tl.rank_genes_groups(
#     adata_ref,
#     groupby="cell_type",
#     groups=["endothelial cell"],
#     method="t-test",
#     key_added="rank_genes_groups_ttest_cell_type",
#     use_raw=False,
#     pts=True,
#     n_genes=None,
# )

In [74]:
# adata_ref.write_h5ad("../breast_cancer/breast_sc_atlas_DE.h5ad")

In [ ]:
df = sc.get.rank_genes_groups_df(
    adata_ref,
    group="endothelial cell",
    key="rank_genes_groups_ttest_cell_type",
    gene_symbols="feature_name",
    log2fc_min=0,
    pval_cutoff=0.05,
)

df.index = df["feature_name"]

df

,names,scores,logfoldchanges,pvals,pvals_adj,feature_name,pct_nz_group,pct_nz_reference
feature_name,,,,,,,,
SPARCL1,ENSG00000152583,225.119476,5.688367,0.000000,0.000000,SPARCL1,0.944610,0.162332
IGFBP7,ENSG00000163453,224.606766,5.611556,0.000000,0.000000,IGFBP7,0.957579,0.240174
IFITM3,ENSG00000142089,218.478165,3.728084,0.000000,0.000000,IFITM3,0.976088,0.539617
IFI27,ENSG00000165949,210.264954,4.375357,0.000000,0.000000,IFI27,0.960686,0.302659
RAMP2,ENSG00000131477,179.356018,7.274539,0.000000,0.000000,RAMP2,0.880438,0.058622
...,...,...,...,...,...,...,...,...
CFAP44,ENSG00000206530,2.045244,0.299712,0.040862,0.048734,CFAP44,0.023237,0.022156
SPIN4,ENSG00000186767,2.039858,0.361974,0.041396,0.049357,SPIN4,0.017833,0.016684
ZNF500,ENSG00000103199,2.037711,0.227460,0.041610,0.049602,ZNF500,0.037287,0.034124


In [14]:
df.head(20).index

CategoricalIndex(['SPARCL1', 'IGFBP7', 'IFITM3', 'IFI27', 'RAMP2', 'PLVAP',
                  'PECAM1', 'GNG11', 'A2M', 'RNASE1', 'ENG', 'VIM', 'CRIP2',
                  'NPDC1', 'VWF', 'AQP1', 'IGFBP4', 'FKBP1A', 'EMCN',
                  'ADGRL4'],
                 categories=['A1BG', 'A1BG-AS1', 'A2M', 'A4GALT', ..., 'ZYG11B', 'ZYX', 'ZZEF1', 'ZZZ3'], ordered=False, dtype='category', name='feature_name')

In [16]:
# adata = ad.read_h5ad(data_dir + data_file)
# adata = ad.read_h5ad(
#     path_to_save +'/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2_nicheVI.h5ad'
# )

adata = ad.read_h5ad(
    path_to_save + "/xenium_breast_cancer_S1_R1_2_nicheVI.h5ad"
)


del adata.uns["log1p"]

print(adata)

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 
'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'sample', 'cell_type', 'n_genes', 
'n_counts', '_scvi_batch', '_scvi_labels', 'cell_type_coarse', 'index', '_scvi_sample'
    var: 'mt'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'cell_type_coarse_colors', 'neighbors', 'pca', 'simvi_train_mask', 
'simvi_val_mask', 'umap'
    obsm: 'X_pca', 'X_scVI', 'X_scVI_MDE', 'X_scanvi', 'X_umap', 'banksy_pc_20_lambda_0.0', 
'banksy_pc_20_lambda_0.2', 'banksy_pc_20_lambda_0.5', 'ln_s10_w400_scanvi_lr0.0005_poisson_X_nicheVI', 
'ln_s20_w400_scanvi_lr0.0005_poisson_X_nicheVI', 'neighborhood_composition', 'niche_distances', 'niche_indexes', 
'qz1_m', 'qz1_m_niche_ct', 'simvi_both', 'simvi_both_08', 'simvi_both_MDE', 'simvi_both_MDE_08', 'simvi_interact', 
'simvi_interact_08', 'simvi_interact_MDE', 'simvi_interact_MDE_08', 'simvi_intrinsic', 'simvi_intrinsic_08', 
'simvi_intrinsic_MDE', 'simvi_intrinsic_MDE_08', 'spatial'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [17]:
common_genes = df.index.intersection(adata.var_names)
common_genes_list = common_genes.tolist()

In [18]:
len(common_genes_list)

74

In [19]:
print(common_genes_list)

[
    'RAMP2',
    'PECAM1',
    'VWF',
    'AQP1',
    'EGFL7',
    'CLEC14A',
    'TCF4',
    'CAV1',
    'CLDN5',
    'CD93',
    'MMRN2',
    'CAVIN2',
    'PDK4',
    'SOX17',
    'BACE2',
    'NOSTRIN',
    'ANKRD29',
    'RAPGEF3',
    'STC1',
    'KDR',
    'SOX18',
    'HOXD9',
    'FLNB',
    'EDN1',
    'ANGPT2',
    'HOXD8',
    'ZEB1',
    'BTNL9',
    'SNAI1',
    'PPARG',
    'CXCL12',
    'TCIM',
    'CD79B',
    'MYLK',
    'CD9',
    'DST',
    'EDNRB',
    'MMP2',
    'CCPG1',
    'SVIL',
    'CTTN',
    'ESM1',
    'PELI1',
    'ANKRD28',
    'CSF3',
    'OCIAD2',
    'MAP3K8',
    'DUSP5',
    'CDC42EP1',
    'SLC25A37',
    'DAPK3',
    'TIFA',
    'MEDAG',
    'CRHBP',
    'TIMP4',
    'JUP',
    'ALDH1A3',
    'FBLIM1',
    'LGALSL',
    'TCF15',
    'VOPP1',
    'ENAH',
    'TRIB1',
    'POSTN',
    'ZNF562',
    'PDCD1LG2',
    'USP53',
    'TMEM147',
    'MMP1',
    'PTGDS',
    'LPL',
    'AKR1C1',
    'TRAPPC3',
    'NCAM1'
]

In [20]:
confident_genes_proseg = [
    "ESM1",
    "KDR",
    "TCIM",
    "NDUFA4L2",
    "SNAI1",
    "CDC42EP1",
    "ZEB1",
    "DAPK3",
]

confident_genes_proseg_08 = [
    "ESM1",
    "OXTR",
    "PRDM1",
    "KDR",
    "TCIM",
    "NDUFA4L2",
    "SNAI1",
    "CDC42EP1",
    "ZEB1",
    "DAPK3",
]

all_genes_proseg = [
    "ESM1",
    "TCIM",
    "DUSP5",
    "CDC42EP1",
    "PRDM1",
    "VOPP1",
    "SQLE",
    "TRAF4",
    "PCLAF",
    "DNTTIP1",
    "CCND1",
    "KDR",
    "ZEB1",
    "SNAI1",
    "OXTR",
    "DAPK3",
    "SEC24A",
    "NDUFA4L2",
    "CENPF",
]

confident_genes_xenium = [
    "ESM1",
    "KDR",
    "PRDM1",
    "SNAI1",
    "NDUFA4L2",
    "RAMP2",
    "CDC42EP1",
    "ZEB1",
]

all_genes_xenium = [
    "ESM1",
    "FOXA1",
    "PCLAF",
    "KRT7",
    "TOP2A",
    "CENPF",
    "SERHL2",
    "ABCC11",
    "GJB2",
    "TFAP2A",
    "DMKN",
    "MKI67",
    "SCD",
    "KDR",
    "OXTR",
    "HMGA1",
    "PRDM1",
    "FASN",
    "TCIM",
    "GATA3",
    "SNAI1",
    "CCND1",
    "MYO5B",
    "NDUFA4L2",
    "TRAF4",
    "SQLE",
    "JUP",
    "CD9",
    "SMS",
    "RAMP2",
    "DUSP5",
    "CDC42EP1",
    "PTRHD1",
    "OCIAD2",
    "EIF4EBP1",
    "VOPP1",
    "SH3YL1",
    "ZEB1",
    "CCDC6",
    "TUBA4A",
    "SEC24A",
    "MDM2",
    "DAPK3",
    "ADAM9",
    "SRPK1",
    "NARS",
    "DNTTIP1",
    "POLR2J3",
    "THAP2",
    "CTTN",
    "ENAH",
]

In [21]:
def compute_metrics(ground_truth, de_genes):
    ground_truth_set = set(ground_truth)
    de_genes_set = set(de_genes)

    TP = len(ground_truth_set & de_genes_set)  # Intersection
    FP = len(de_genes_set - ground_truth_set)  # Found but not in ground truth
    print(de_genes_set - ground_truth_set)
    FN = len(ground_truth_set - de_genes_set)  # In ground truth but not found

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {"precision": precision, "recall": recall, "F1-score": f1}

In [83]:
compute_metrics(common_genes_list, confident_genes_proseg_08)

{'PRDM1', 'NDUFA4L2', 'OXTR'}

{'precision': 0.7,
 'recall': 0.09333333333333334,
 'F1-score': 0.16470588235294115}

In [84]:
compute_metrics(common_genes_list, confident_genes_proseg)

{'NDUFA4L2'}

{'precision': 0.875,
 'recall': 0.09333333333333334,
 'F1-score': 0.1686746987951807}

In [85]:
compute_metrics(common_genes_list, all_genes_proseg)

{'CENPF', 'DNTTIP1', 'PRDM1', 'SQLE', 'CCND1', 'SEC24A', 'PCLAF', 'TRAF4', 'OXTR', 'NDUFA4L2'}

{'precision': 0.47368421052631576,
 'recall': 0.12,
 'F1-score': 0.19148936170212763}

In [86]:
compute_metrics(common_genes_list, confident_genes_xenium)

{'PRDM1', 'NDUFA4L2'}

{'precision': 0.75, 'recall': 0.08, 'F1-score': 0.14457831325301204}

In [87]:
compute_metrics(common_genes_list, all_genes_xenium)

{
    'DNTTIP1',
    'MKI67',
    'SRPK1',
    'FASN',
    'SQLE',
    'FOXA1',
    'PCLAF',
    'GATA3',
    'TUBA4A',
    'MDM2',
    'HMGA1',
    'CCND1',
    'EIF4EBP1',
    'MYO5B',
    'ADAM9',
    'CENPF',
    'PRDM1',
    'SEC24A',
    'POLR2J3',
    'TRAF4',
    'ABCC11',
    'NDUFA4L2',
    'SMS',
    'CCDC6',
    'THAP2',
    'KRT7',
    'PTRHD1',
    'DMKN',
    'TFAP2A',
    'TOP2A',
    'OXTR',
    'SH3YL1',
    'GJB2',
    'NARS',
    'SCD',
    'SERHL2'
}

{'precision': 0.29411764705882354,
 'recall': 0.2,
 'F1-score': 0.2380952380952381}

In [22]:
confident_ttest_proseg = [
    "ESM1",
    "TCIM",
    "KDR",
    "PRDM1",
    "RAMP2",
    "SNAI1",
    "SOX18",
    "NDUFA4L2",
    "FBLIM1",
    "IL3RA",
    "ZEB1",
    "DUSP5",
    "JUP",
    "CDC42EP1",
    "VOPP1",
    "FOXC2",
    "STC1",
    "MMP2",
    "AVPR1A",
    "ADGRE5",
    "DAPK3",
    "PDGFRB",
    "EIF4EBP1",
    "EDNRB",
]

all_ttest_proseg = [
    "ESM1",
    "TCIM",
    "KDR",
    "PRDM1",
    "CCND1",
    "RAMP2",
    "PCLAF",
    "CLEC14A",
    "SNAI1",
    "CENPF",
    "SOX18",
    "NDUFA4L2",
    "FBLIM1",
    "IL3RA",
    "ZEB1",
    "FOXA1",
    "DUSP5",
    "POLR2J3",
    "JUP",
    "CDC42EP1",
    "TOP2A",
    "TRAF4",
    "OXTR",
    "VOPP1",
    "FOXC2",
    "STC1",
    "ERBB2",
    "MMP2",
    "AVPR1A",
    "ADGRE5",
    "MKI67",
    "SQLE",
    "SEC24A",
    "DAPK3",
    "CD9",
    "PDGFRB",
    "FAM107B",
    "TFAP2A",
    "ADAM9",
    "EIF4EBP1",
    "CD93",
    "RUNX1",
    "EDNRB",
    "SMS",
    "GJB2",
    "SERHL2",
]

all_ttest_xenium = [
    "ESM1",
    "FOXA1",
    "CCND1",
    "KRT7",
    "TCIM",
    "KDR",
    "SCD",
    "FASN",
    "PCLAF",
    "GATA3",
    "RAMP2",
    "SERHL2",
    "CLEC14A",
    "PRDM1",
    "ABCC11",
    "TOP2A",
    "EPCAM",
    "ERBB2",
    "TRAF4",
    "SMS",
    "CENPF",
    "MYO5B",
    "CD9",
    "SQLE",
    "JUP",
    "KRT8",
    "TFAP2A",
    "S100A14",
    "HMGA1",
    "AR",
    "MDM2",
    "MKI67",
    "SNAI1",
    "RUNX1",
    "EIF4EBP1",
    "DUSP5",
    "CXCR4",
    "ZEB1",
    "SEC24A",
    "CDC42EP1",
    "OXTR",
    "TENT5C",
    "KLF5",
    "VOPP1",
    "CCDC6",
    "NARS",
    "FAM107B",
    "RHOH",
    "ANKRD30A",
    "ELF3",
    "APOC1",
    "BTNL9",
    "AVPR1A",
    "POLR2J3",
    "NDUFA4L2",
    "CDH1",
    "CD93",
    "ADGRE5",
    "TUBA4A",
    "ADAM9",
    "MLPH",
    "EDNRB",
    "LYPD3",
    "PDGFRB",
    "IL3RA",
    "FBLIM1",
    "STC1",
    "PTRHD1",
    "FOXC2",
    "DMKN",
    "TPD52",
    "SH3YL1",
    "DAPK3",
    "SOX18",
    "SRPK1",
    "CLDN4",
    "GJB2",
    "PELI1",
    "FCGR3A",
    "CCPG1",
    "C6orf132",
    "SLC5A6",
    "THAP2",
    "QARS",
    "CTTN",
    "ENAH",
]

confident_ttest_xenium = [
    "ESM1",
    "TCIM",
    "KDR",
    "RAMP2",
    "CLEC14A",
    "PRDM1",
    "JUP",
    "SNAI1",
    "DUSP5",
    "ZEB1",
    "CDC42EP1",
    "OXTR",
    "BTNL9",
    "AVPR1A",
    "NDUFA4L2",
    "CD93",
    "ADGRE5",
    "EDNRB",
    "PDGFRB",
    "IL3RA",
    "FBLIM1",
    "STC1",
    "FOXC2",
    "SOX18",
    "PELI1",
    "CCPG1",
]

In [27]:
all_cside_xenium = [
    "CCND1",
    "KDR",
    "FOXA1",
    "TCIM",
    "KRT7",
    "ESM1",
    "SCD",
    "FASN",
    "RAMP2",
    "ERBB2",
    "CLEC14A",
    "GATA3",
    "PRDM1",
    "POLR2J3",
    "SMS",
    "EPCAM",
    "PCLAF",
    "KRT8",
    "SERHL2",
    "TOP2A",
    "MDM2",
    "ABCC11",
    "JUP",
    "TRAF4",
    "SQLE",
    "CD9",
    "MYO5B",
    "NARS",
    "RUNX1",
    "CENPF",
    "CCDC6",
    "VOPP1",
    "S100A14",
    "ZEB1",
    "HMGA1",
    "TFAP2A",
    "AR",
    "ADAM9",
    "SEC24A",
    "EIF4EBP1",
    "MKI67",
    "TUBA4A",
    "SNAI1",
    "PDGFRB",
    "CXCR4",
    "CDC42EP1",
    "ANKRD30A",
    "APOC1",
    "FBLIM1",
    "CTTN",
    "DUSP5",
    "KLF5",
    "TENT5C",
    "IL3RA",
    "TPD52",
    "AVPR1A",
    "FAM107B",
    "RHOH",
    "OXTR",
    "ADGRE5",
    "MLPH",
    "PTRHD1",
    "SH3YL1",
    "ELF3",
    "CD93",
    "EDNRB",
    "SRPK1",
    "ENAH",
    "CDH1",
    "CCPG1",
    "LYPD3",
    "NDUFA4L2",
    "STC1",
    "PELI1",
    "QARS",
    "FOXC2",
    "DAPK3",
    "SOX18",
    "BTNL9",
    "GJB2",
    "TMEM147",
    "CLDN4",
    "FCGR3A",
    "ACTA2",
    "DMKN",
    "SLC5A6",
    "THAP2",
    "LDHB",
    "SERPINB9",
    "POSTN",
    "TCF4",
    "C6orf132",
    "DNTTIP1",
    "OCIAD2",
    "SEC11C",
    "LGALSL",
    "SVIL",
    "RTKN2",
    "NPM3",
    "HDC",
    "BASP1",
    "ITGAX",
    "PTN",
]

all_cside_proseg = [
    "KDR",
    "ESM1",
    "TCIM",
    "RAMP2",
    "CCND1",
    "CLEC14A",
    "PRDM1",
    "POLR2J3",
    "ZEB1",
    "VOPP1",
    "FBLIM1",
    "IL3RA",
    "JUP",
    "PCLAF",
    "SNAI1",
    "CD93",
    "SOX18",
    "DUSP5",
    "TCF4",
    "TRAF4",
    "TOP2A",
    "CDC42EP1",
    "SEC24A",
    "ERBB2",
    "FOXA1",
    "CENPF",
    "NDUFA4L2",
    "CCPG1",
    "SQLE",
    "MMP2",
    "ADGRE5",
    "ADAM9",
    "AVPR1A",
    "TUBA4A",
    "CTTN",
    "FOXC2",
    "CD9",
    "STC1",
    "PDGFRB",
    "SVIL",
    "MKI67",
    "EIF4EBP1",
    "PELI1",
    "OXTR",
    "SMS",
    "DNTTIP1",
    "EDNRB",
    "FAM107B",
    "NARS",
    "CCDC6",
    "MDM2",
    "ACTA2",
    "RUNX1",
    "DAPK3",
    "LDHB",
    "LGALSL",
    "CXCR4",
    "PTRHD1",
    "TRAPPC3",
    "MYLK",
    "HDC",
    "TFAP2A",
    "QARS",
    "SRPK1",
    "GJB2",
    "SCD",
    "PECAM1",
    "TMEM147",
    "ZNF562",
    "SERHL2",
    "SERPINB9",
    "VWF",
    "KRT7",
    "ANKRD28",
    "PTN",
    "ABCC11",
    "FASN",
    "THAP2",
    "LRRC15",
]

In [28]:
len(all_cside_proseg)

79

In [29]:
compute_metrics(common_genes_list, all_cside_proseg)

{
    'PDGFRB',
    'SCD',
    'SMS',
    'ERBB2',
    'LDHB',
    'SQLE',
    'GJB2',
    'IL3RA',
    'QARS',
    'PTN',
    'FOXA1',
    'ACTA2',
    'TUBA4A',
    'RUNX1',
    'PCLAF',
    'OXTR',
    'KRT7',
    'FAM107B',
    'CCDC6',
    'ADAM9',
    'THAP2',
    'SERPINB9',
    'TRAF4',
    'ADGRE5',
    'FOXC2',
    'HDC',
    'FASN',
    'AVPR1A',
    'MDM2',
    'LRRC15',
    'CENPF',
    'SEC24A',
    'EIF4EBP1',
    'CCND1',
    'SERHL2',
    'POLR2J3',
    'ABCC11',
    'CXCR4',
    'SRPK1',
    'NARS',
    'TFAP2A',
    'PTRHD1',
    'MKI67',
    'DNTTIP1',
    'PRDM1',
    'NDUFA4L2',
    'TOP2A'
}

{'precision': 0.4050632911392405,
 'recall': 0.43243243243243246,
 'F1-score': 0.41830065359477125}

In [26]:
compute_metrics(common_genes_list, all_cside_xenium)

{
    'PDGFRB',
    'SCD',
    'SMS',
    'ERBB2',
    'MYO5B',
    'LDHB',
    'GJB2',
    'SH3YL1',
    'ELF3',
    'IL3RA',
    'C6orf132',
    'HMGA1',
    'FOXA1',
    'SLC5A6',
    'ACTA2',
    'QARS',
    'PTN',
    'TUBA4A',
    'SEC11C',
    'RUNX1',
    'PCLAF',
    'AR',
    'OXTR',
    'KRT8',
    'KRT7',
    'KLF5',
    'FAM107B',
    'CCDC6',
    'ADAM9',
    'DMKN',
    'GATA3',
    'THAP2',
    'SERPINB9',
    'CDH1',
    'TRAF4',
    'ADGRE5',
    'RHOH',
    'PRDM1',
    'FCGR3A',
    'FOXC2',
    'BASP1',
    'ANKRD30A',
    'FASN',
    'HDC',
    'AVPR1A',
    'TPD52',
    'MDM2',
    'SQLE',
    'NPM3',
    'CENPF',
    'RTKN2',
    'SEC24A',
    'SERHL2',
    'APOC1',
    'CCND1',
    'EIF4EBP1',
    'LYPD3',
    'TENT5C',
    'CXCR4',
    'SRPK1',
    'NARS',
    'CLDN4',
    'POLR2J3',
    'ABCC11',
    'TFAP2A',
    'ITGAX',
    'S100A14',
    'PTRHD1',
    'MKI67',
    'DNTTIP1',
    'MLPH',
    'NDUFA4L2',
    'TOP2A',
    'EPCAM'
}

{'precision': 0.2815533980582524,
 'recall': 0.3918918918918919,
 'F1-score': 0.327683615819209}

In [89]:
compute_metrics(common_genes_list, confident_ttest_proseg)

{'AVPR1A', 'PDGFRB', 'IL3RA', 'FOXC2', 'PRDM1', 'EIF4EBP1', 'NDUFA4L2', 'ADGRE5'}

{'precision': 0.6666666666666666,
 'recall': 0.21333333333333335,
 'F1-score': 0.3232323232323232}

In [90]:
compute_metrics(common_genes_list, all_ttest_proseg)

{
    'PDGFRB',
    'MKI67',
    'RUNX1',
    'SQLE',
    'FOXA1',
    'PCLAF',
    'ADGRE5',
    'AVPR1A',
    'IL3RA',
    'CCND1',
    'EIF4EBP1',
    'ADAM9',
    'CENPF',
    'PRDM1',
    'SEC24A',
    'POLR2J3',
    'TRAF4',
    'FAM107B',
    'NDUFA4L2',
    'SMS',
    'FOXC2',
    'TFAP2A',
    'TOP2A',
    'OXTR',
    'GJB2',
    'ERBB2',
    'SERHL2'
}

{'precision': 0.41304347826086957,
 'recall': 0.25333333333333335,
 'F1-score': 0.3140495867768595}

In [91]:
compute_metrics(common_genes_list, all_ttest_xenium)

{
    'PDGFRB',
    'EPCAM',
    'KRT8',
    'MKI67',
    'CXCR4',
    'ELF3',
    'SRPK1',
    'FASN',
    'TENT5C',
    'RUNX1',
    'SQLE',
    'FOXA1',
    'PCLAF',
    'LYPD3',
    'ADGRE5',
    'GATA3',
    'C6orf132',
    'AVPR1A',
    'TUBA4A',
    'GJB2',
    'MDM2',
    'IL3RA',
    'TPD52',
    'HMGA1',
    'CCND1',
    'QARS',
    'EIF4EBP1',
    'RHOH',
    'MYO5B',
    'ADAM9',
    'CENPF',
    'SLC5A6',
    'CDH1',
    'FCGR3A',
    'CLDN4',
    'SEC24A',
    'S100A14',
    'PRDM1',
    'POLR2J3',
    'TRAF4',
    'FAM107B',
    'ABCC11',
    'MLPH',
    'NDUFA4L2',
    'SMS',
    'THAP2',
    'CCDC6',
    'KRT7',
    'PTRHD1',
    'DMKN',
    'FOXC2',
    'APOC1',
    'KLF5',
    'AR',
    'TFAP2A',
    'TOP2A',
    'SH3YL1',
    'NARS',
    'ANKRD30A',
    'OXTR',
    'ERBB2',
    'SCD',
    'SERHL2'
}

{'precision': 0.26744186046511625,
 'recall': 0.30666666666666664,
 'F1-score': 0.2857142857142857}

In [92]:
compute_metrics(common_genes_list, confident_ttest_xenium)

{'AVPR1A', 'PDGFRB', 'IL3RA', 'FOXC2', 'PRDM1', 'OXTR', 'NDUFA4L2', 'ADGRE5'}

{'precision': 0.6923076923076923,
 'recall': 0.24,
 'F1-score': 0.3564356435643564}

In [93]:
compute_metrics(common_genes_list, confident_ttest_proseg)

{'AVPR1A', 'PDGFRB', 'IL3RA', 'FOXC2', 'PRDM1', 'EIF4EBP1', 'NDUFA4L2', 'ADGRE5'}

{'precision': 0.6666666666666666,
 'recall': 0.21333333333333335,
 'F1-score': 0.3232323232323232}